IMPORTING MODULES

In [ ]:

import os
import numpy as np
import mne
from mne.time_frequency import psd_array_welch
from antropy import hjorth_params
from antropy.entropy import spectral_entropy, perm_entropy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

PREPROCESSING AND FEATURE EXTRACTION

In [ ]:
def preprocess_edf(file_path, window_sec=5):
    try:
        raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
        raw.pick_types(eeg=True)

        montage = mne.channels.make_standard_montage("standard_1020")
        raw.set_montage(montage, on_missing="ignore")

        standard_channels = [
            "Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2",
            "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"
        ]

        present_channels = [ch for ch in standard_channels if ch in raw.ch_names]
        missing_channels = [ch for ch in standard_channels if ch not in raw.ch_names]

        raw.pick_channels(present_channels)
        if missing_channels:
            zero_data = np.zeros((len(missing_channels), raw.n_times))
            info_missing = mne.create_info(missing_channels, raw.info['sfreq'], ch_types="eeg")
            raw.add_channels([mne.io.RawArray(zero_data, info_missing)])
        raw.reorder_channels(standard_channels)

        data, _ = raw.get_data(return_times=True)
        sfreq = raw.info["sfreq"]
        n_channels, n_samples = data.shape

        window_size = int(window_sec * sfreq)
        n_windows = n_samples // window_size

        bands = [(0.5, 4), (4, 8), (8, 12), (12, 16),
                 (16, 25), (25, 30), (30, 40)]
        band_indices = None

        features = []
        for w in range(n_windows):
            start = w * window_size
            stop = start + window_size
            window_data = data[:, start:stop]

            psd, freqs = psd_array_welch(
                window_data, sfreq=sfreq,
                fmin=0.5, fmax=40, n_fft=256, n_jobs=1
            )

            if band_indices is None:
                band_indices = [
                    np.logical_and(freqs >= fmin, freqs <= fmax)
                    for fmin, fmax in bands
                ]

            psd_bands = np.stack(
                [psd[:, idx].mean(axis=1) for idx in band_indices], axis=1
            )

            window_features = []
            for ch in range(n_channels):
                x = window_data[ch]
                ch_psd = psd_bands[ch].tolist()
                hjorth = hjorth_params(x)
                spec_ent = spectral_entropy(x, sfreq, method="welch", normalize=True)
                perm_ent = perm_entropy(x, normalize=True)
                ch_features = ch_psd + list(hjorth) + [spec_ent, perm_ent]
                window_features.extend(ch_features)

            features.append(window_features)

        return np.array(features)

    except Exception:
        return None

DATASET

In [ ]:
root_folder = "TUH_EEG"
folders = [("00_epilepsy", 0), ("01_no_epilepsy", 1)]

In [ ]:
all_X, all_y = [], []

for folder_name, label in folders:
    folder_path = os.path.join(root_folder, folder_name)
    count = 0
    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            if f.lower().endswith(".edf"):
                file_path = os.path.join(dirpath, f)
                X_trial = preprocess_edf(file_path, window_sec=5)
                if X_trial is not None:
                    all_X.append(X_trial)
                    all_y.append(label)
                    count += 1
                if count >= 200:
                    break
        if count >= 200:
            break


In [ ]:
if len(all_X) > 0:
    cleaned_X, cleaned_y = [], []
    for x, label in zip(all_X, all_y):
        if x is not None and x.shape[0] > 0:
            cleaned_X.append(x)
            cleaned_y.append(label)

    if len(cleaned_X) == 0:
        raise ValueError("No valid trials")

    max_windows = max(x.shape[0] for x in cleaned_X)
    n_features = cleaned_X[0].shape[1]

    X_final = np.zeros((len(cleaned_X), max_windows, n_features))
    valid_lengths = []
    for i, x in enumerate(cleaned_X):
        X_final[i, :x.shape[0], :] = x
        valid_lengths.append(x.shape[0])

    y = np.array(cleaned_y)

NORMALIZATION

In [ ]:
mean = X_final.mean(axis=(0, 1), keepdims=True)
std = X_final.std(axis=(0, 1), keepdims=True) + 1e-8
X = (X_final - mean) / std
X = np.nan_to_num(X, nan=0.0)

SHUFFLING

In [ ]:
indices = np.arange(len(X))
np.random.shuffle(indices)
X = X[indices]
y = y[indices]
valid_lengths = np.array(valid_lengths)[indices]

In [ ]:
np.save("EEG_features_5s.npy", X)
np.save("EEG_labels_5s.npy", y)
np.save("EEG_lengths_5s.npy", valid_lengths)

TRANSFORMER MODEL

In [ ]:
class Time2Vec(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.w0 = nn.Parameter(torch.randn(1, 1))
        self.b0 = nn.Parameter(torch.randn(1, 1))
        self.w = nn.Parameter(torch.randn(1, d_model - 1))
        self.b = nn.Parameter(torch.randn(1, d_model - 1))

    def forward(self, t):
        linear = self.w0 * t + self.b0
        periodic = torch.sin(self.w * t + self.b)
        return torch.cat([linear, periodic], dim=-1)


In [ ]:
class EEGTransformer(nn.Module):
    def __init__(self, feature_dim, d_model=128, n_head=8, hidden_dim=512, num_layers=4, drop_prob=0.5):
        super().__init__()
        self.input_proj = nn.Linear(feature_dim, d_model)
        self.time2vec = Time2Vec(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_head, dim_feedforward=hidden_dim,
            dropout=drop_prob, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.classifier = nn.Linear(d_model, 1)

    def forward(self, x, mask=None):
        b, t, _ = x.shape
        x = self.input_proj(x)
        t_idx = torch.arange(t, device=x.device).view(1, t, 1).expand(b, t, 1).float()
        x = x + self.time2vec(t_idx)
        x = self.encoder(x, src_key_padding_mask=mask)
        x = x.mean(dim=1)
        return self.classifier(x)

LOADING AND SPLITING DATA

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                        torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)),
                          batch_size=8, shuffle=True)

test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                       torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)),
                         batch_size=8)


MODEL TRAINING

In [ ]:
def train_model(model, train_loader, device, epochs=40, lr=1e-3):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        train_loss, train_acc = 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_acc += (torch.sigmoid(logits).round() == y_batch).float().mean().item()

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss/len(train_loader):.4f}, Acc: {train_acc/len(train_loader):.4f}")

EVALUATION

In [ ]:
def test_model(model, test_loader, device):
    model.eval()
    test_acc = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            test_acc += (torch.sigmoid(logits).round() == y_batch).float().mean().item()
    print(f"Test Accuracy: {test_acc/len(test_loader):.4f}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EEGTransformer(feature_dim=X.shape[2]).to(device)
train_model(model, train_loader, device)
test_model(model, test_loader, device)


SAVING MODEL

In [ ]:
torch.save(model.state_dict(), "epilepsy.h5")